# Demo LLM Group 4
Chạy prepare_data và train trước. Notebook gọi src, không viết lại model. GPT-2 là phần tùy chọn cần tải weights.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
os.chdir(root)
sys.path.insert(0, str(root))
import torch
torch.set_num_threads(1)
from src.train import load_checkpoint
from src.generate import generate
model, tok, payload = load_checkpoint('checkpoints/best.pt')
print('Parameters:', model.parameter_count(), 'Expected:', model.expected_parameter_count())

In [ ]:
prompt = 'ROMEO:\n'
ids = torch.tensor([tok.encode(prompt)])
print('IDs:', ids, 'logits:', model(ids).shape)
print(generate(model, tok, prompt, strategy='greedy')['text'])

In [ ]:
from src.visualize import plot_loss, plot_attention
plot_loss('outputs/metrics/training.csv', 'outputs/figures/loss.png')
model.eval()
with torch.no_grad():
    x = model.token_embedding(ids) + model.position_embedding(torch.arange(ids.shape[1]))
    _, weights = model.blocks[0].attn(model.blocks[0].ln1(x), return_weights=True)
plot_attention(weights[0, 0], list(prompt), 'outputs/figures/attention.png')

## Q2 / Q3
Chạy `python -m scripts.run_experiments` để lưu Q2. Thêm `--with-pretrained` để chạy Q3. Không so PPL giữa tokenizer khác nhau.